# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets and their fields along with their `@id` values.

In [ ]:
# List all available record sets and fields with their @id
print("Available Record Sets and Fields:")
for rs in dataset.metadata.record_sets:
    print(f"Record set: {rs['@id']} | Name: {rs.get('name', '<no name>')}")
    print("\tFields (by @id):")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"\t- {field.get('@id', '<no id>')} | {field.get('name', '<no name>')}")
        else:
            print(f"\t- {field}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record_set @id values
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records for record_set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records. Columns:", df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records based on criteria, normalizing numeric fields, and grouping by attributes. All column access is via field `@id`.

In [ ]:
# Select one record set for demonstration
# Replace with the desired record set @id as necessary
record_set_id = record_sets_ids[0]
df = dataframes[record_set_id]

print(f"Working with record set {record_set_id}")
print("Columns:", df.columns.tolist())

# Find a numeric field
numeric_field = None
for c in df.columns:
    # Try select a numeric-looking field (float or integer columns)
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_field = c
        break
if numeric_field is None:
    print("No numeric field found in this record set.")
else:
    print(f"Using numeric field: {numeric_field}")

    # Drop missing values just in case
    dff = df.dropna(subset=[numeric_field])

    threshold = dff[numeric_field].mean()
    filtered_df = dff[dff[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > mean ({threshold}):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a non-numeric field
    group_field = None
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]) and c != numeric_field:
            group_field = c
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field or its relationship to another field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: Histogram or boxplot of the numeric field
if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field} in record set {record_set_id}')
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        top_groups = df[group_field].value_counts().nlargest(5).index
        sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field} (top 5 groups)')
        plt.show()

## 6. Conclusion
This notebook loaded the dataset with `mlcroissant`, reviewed its record sets and fields using unique `@id` identifiers, extracted one record set for analysis, cleaned and normalized a numeric field, and visualized the data. You can now proceed to deeper domain-specific analytics as needed.